
# Gradio Text→Video Demo

<img src="https://cdn-icons-png.flaticon.com/512/3447/3447651.png" alt="prompt" height="50"/>


<img src="https://cdn-icons-png.flaticon.com/512/1179/1179069.png" alt="prompt" height="50"/>


This template demonstrates how to generate short video clips from text prompts using a powerful diffusion-based video generation model.

The generated videos are rendered and immediately previewed inside the notebook, making it ideal for experimentation or creative prototyping.

On [Saturn Cloud](https://saturncloud.io), you can scale from a single NVIDIA GPU to multi-GPU clusters, enabling distributed inference for larger models or higher throughput workloads — all within a managed, GPU-ready environment.

Let's install the dependencies.

In [ ]:
!pip install -q gradio>=4.0.0 diffusers>=0.25.0 transformers>=4.36.0 accelerate>=0.25.0 torch>=2.0.0 torchvision>=0.15.0 opencv-python>=4.8.0 imageio>=2.33.0 imageio-ffmpeg>=0.4.9 pillow>=10.0.0

This section initializes the core components of the text-to-video generation pipeline. We use the cerspense/zeroscope_v2_576w model — a powerful diffusion model fine-tuned for higher-resolution video generation at 576×320 pixels. It's designed to turn natural language prompts into short, animated video clips.

In [ ]:
import gradio as gr
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Model configuration
MODEL_ID = "cerspense/zeroscope_v2_576w"  # Higher resolution: 576x320
# Alternative: "damo-vilab/text-to-video-ms-1.7b" for 256x256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🖥️  Using device: {DEVICE}")
print(f"📦 Model: {MODEL_ID}")
print(f"📺 Output resolution: 576x320 pixels")

This section handles loading the text-to-video diffusion model using Hugging Face's DiffusionPipeline. This ensure the model is initialized only once, using a global pipeline object to avoid repeated downloads or memory spikes.

In [ ]:
# Global pipeline variable
pipe = None

def initialize_model():
    """Initialize the text-to-video diffusion model"""
    global pipe

    if pipe is None:
        print("📥 Loading text-to-video model...")
        pipe = DiffusionPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            # Note: Not using variant parameter as this model doesn't have fp16 files
        )

        # Optimize with DPM Solver for faster generation
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        pipe = pipe.to(DEVICE)

        # Enable memory optimizations
        if DEVICE == "cuda":
            pipe.enable_model_cpu_offload()
            pipe.enable_vae_slicing()

        print("✅ Model loaded successfully!")

    return pipe

# Initialize the model
model = initialize_model()


This function transforms a text description into a short video clip using the loaded diffusion pipeline. It handles every part of the process — from inference to frame formatting and final export.

In [ ]:
def generate_video(prompt, num_inference_steps=25, guidance_scale=9.0, num_frames=16):
    """
    Generate a video from a text prompt

    Args:
        prompt: Text description of the video
        num_inference_steps: Number of denoising steps (higher = better quality, slower)
        guidance_scale: How closely to follow the prompt (7-12 recommended)
        num_frames: Number of frames to generate (16 frames ≈ 5 seconds at 3fps)

    Returns:
        Path to generated video file
    """
    try:
        # Generate video frames
        print(f"🎬 Generating video for: '{prompt}'")
        video_output = pipe(
            prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            num_frames=num_frames,
            height=320,  # Zeroscope native resolution
            width=576,   # Zeroscope native resolution
        )

        # Extract frames properly - the output is [batch, frames, channels, height, width]
        video_frames = video_output.frames[0]  # Get first batch

        # Convert to proper format for export_to_video
        # Frames should be a list of numpy arrays with shape (height, width, 3)
        formatted_frames = []
        for frame in video_frames:
            # Convert from tensor/array to numpy if needed
            if torch.is_tensor(frame):
                frame = frame.cpu().numpy()

            # Ensure frame is in correct format (H, W, C)
            if frame.ndim == 3:
                # If channels first (C, H, W), transpose to (H, W, C)
                if frame.shape[0] == 3 or frame.shape[0] == 1:
                    frame = np.transpose(frame, (1, 2, 0))

            # Ensure values are in 0-255 range for uint8
            if frame.dtype == np.float32 or frame.dtype == np.float64:
                frame = (frame * 255).astype(np.uint8)

            formatted_frames.append(frame)

        # Export to video file
        output_path = "/tmp/generated_video.mp4"
        video_path = export_to_video(formatted_frames, output_path, fps=3)

        print(f"✅ Video generated: {video_path}")
        return video_path

    except Exception as e:
        print(f"❌ Error generating video: {str(e)}")
        import traceback
        traceback.print_exc()
        raise gr.Error(f"Failed to generate video: {str(e)}")

his section builds a user-friendly interface to generate video clips directly from natural language prompts. The app is powered by Gradio, which lets you interact with the model in real time — all within the notebook or as a shareable web app.

In [ ]:
# Create Gradio interface
with gr.Blocks(title="Text-to-Video Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🎬 Text→Video Generator

        Generate 5-second video clips from text prompts using diffusion models.
        Simply describe what you want to see, and AI will create it!
        """
    )

    with gr.Row():
        with gr.Column():
            # Input controls
            prompt_input = gr.Textbox(
                label="Video Prompt",
                placeholder="A cat playing with a ball of yarn...",
                lines=3
            )

            with gr.Accordion("Advanced Settings", open=False):
                steps_slider = gr.Slider(
                    minimum=10,
                    maximum=50,
                    value=25,
                    step=5,
                    label="Inference Steps (higher = better quality)",
                )

                guidance_slider = gr.Slider(
                    minimum=1.0,
                    maximum=15.0,
                    value=9.0,
                    step=0.5,
                    label="Guidance Scale (how closely to follow prompt)",
                )

                frames_slider = gr.Slider(
                    minimum=8,
                    maximum=24,
                    value=16,
                    step=8,
                    label="Number of Frames",
                )

            generate_btn = gr.Button("🎥 Generate Video", variant="primary", size="lg")

            # Example prompts
            gr.Examples(
                examples=[
                    ["A cat playing with a ball of yarn on a wooden floor"],
                    ["Waves crashing on a beach at sunset"],
                    ["A coffee cup steaming on a table, camera zoom in"],
                    ["Fireworks exploding in the night sky"],
                    ["A flower blooming in timelapse"],
                ],
                inputs=prompt_input,
            )

        with gr.Column():
            # Output video
            video_output = gr.Video(
                label="Generated Video",
                height=400,
            )

            gr.Markdown(
                """
                ### Tips for better results:
                - Be specific and descriptive
                - Include camera movements (zoom, pan, etc.)
                - Mention lighting and atmosphere
                - Keep prompts clear and concise
                """
            )

    # Connect generate button to function
    generate_btn.click(
        fn=generate_video,
        inputs=[prompt_input, steps_slider, guidance_slider, frames_slider],
        outputs=video_output,
    )

    gr.Markdown(
        """
        ---
        **Note:** Video generation may take 1-3 minutes depending on your hardware.
        """
    )

# Launch the interface
# Display inline within the Jupyter notebook
demo.launch(
    inline=True,   # Display interface directly in the notebook (inline)
    share=False,   # Set to True to create a public gradio.live link
    debug=True,
    height=800,    # Height of the inline iframe for better visibility
)

**Note**: After generating the image withing the gradio section above, stop the code in the cell and run the code block below to see the generated video. This should be repeated after every generated video.

Once the video has been successfully generated and saved (typically to a temporary path like /tmp/generated_video.mp4), you can preview it directly within the notebook using IPython’s built-in video display tools.



In [ ]:
# After generating the video, run this in a new cell:
from IPython.display import Video

# Display the generated video
Video("/tmp/generated_video.mp4", embed=True, width=512, height=512)

## ✅ Conclusion

In this template, we built a complete **text-to-video generation workflow** using the **Zeroscope Diffusion model** and a custom **Gradio interface** — all running on a GPU-powered environment.

Running this pipeline on **[Saturn Cloud](https://saturncloud.io/)** makes the experience both **powerful and efficient**. With Saturn Cloud, you can:

* Launch a **GPU-accelerated Jupyter Notebook** in seconds — ideal for models like Zeroscope.
* Seamlessly integrate **interactive Gradio apps** into your workflow for real-time experimentation.
* Scale up to **multi-GPU compute clusters** for faster video generation and larger batch processing.
* Save and share your notebooks or deploy the app as a **production-grade API**.

By using this template, you've unlocked a modern, reproducible, and GPU-optimized approach to **text-to-video generation** — perfect for prototyping creative AI applications or deploying custom media generation tools.